# Historical curve data

In [1]:
import pandas as pd
import refinitiv.dataplatform.eikon as ek
import refinitiv.data as rd

rd.open_session()
ek.set_app_key('DEFAULT_CODE_BOOK_APP_KEY')

## Instructions

These instructions are inspired by the answer at https://community.developers.lseg.com/discussion/55278/retrieve-historical-benchmark-curve-data?tab=accepted

Alternative fields can be discovered using the `<Data Item Browser>` tool

1. Use the command `<CURVESRCH>` to find the RIC for the wanted curve
2. Fill in the values below

In [2]:
start_date      = '2000-01-01'                                           # Start date of the historical data interval in YYYY-MM-DD format
end_date        = '2025-12-31'                                           # End date of the historical data interval in YYYY-MM-DD format
# Some RICs:
#    '0#EUBMK='         : European Government Benchmark Yield Curve           | Government Bond Benchmark Curve
#    '0#EUXZ=R'         : Euro Government Bond Zero Curve                     | Government Bond Zero Curve
#    '0#EUAAAGZBMK=ECB' : ECB Euro AAA Government Bond Zero Yield Curve       | Government Bond Zero Curve
#    '0#DEXZ=R'         : Germany Government Bond Zero Curve                  | Government Bond Zero Curve
#    'EURAB3EIRS='      : Euro Annual Bond 3 Month Euribor Interest Rate Swap | Interest Rate Swap Curve
#    'EURIRS'           : Euro Interest Rate Swap
curve_ric       = 'EURIRS'                                               # The RIC for the wanted curve
save_as         = f'{curve_ric}_history_{start_date}_to_{end_date}.csv'  # Name of the file where the historical data are stored
accepted_suffix = '='                                                    # The suffix used to identify valid constituents for the curve. This needs to be checked by the user
tr_yield_dic    = "MIDPRICE"  # MIDYIELD                                 # The 'TR' Data Item Code containing the yield information. This can be checked from Data Item Browser. A 'TR.' suffix will be added to the given value

## Run the script

In [3]:
ric_chain = rd.content.pricing.chain.Definition(name=curve_ric).get_stream()
ric_chain.open(False)

<OpenState.Opened: 'Opened'>

In [4]:
constituents = ric_chain.constituents
constituents = [constituent for constituent in constituents if str(constituent).endswith('=')]

In [5]:
# ric_df, e = ek.get_data([curve_ric], ['DSPLY_NAME', ])  # The field values don't matter. The constituent RICs is what this query is for
# rics = ric_df["Instrument"].tolist()

In [6]:
history_df, e = ek.get_data(constituents, [f'TR.{tr_yield_dic}.Date', f'TR.{tr_yield_dic}'], {"SDate": start_date, "EDate": end_date})
history_df.columns = ["Instrument", "Date", "MidYield"]

In [7]:
history_df.head()

,Instrument,Date,MidYield
0,EURAB6E1Y=,2000-01-03T00:00:00Z,4.015
1,EURAB6E1Y=,2000-01-04T00:00:00Z,4.025
2,EURAB6E1Y=,2000-01-05T00:00:00Z,4.045
3,EURAB6E1Y=,2000-01-06T00:00:00Z,4.015
4,EURAB6E1Y=,2000-01-07T00:00:00Z,4.005


In [8]:
# Clean up the data
history_df["Date"]  = [row["Date"][:10] for i, row in history_df.iterrows()]

In [9]:
history_df.drop_duplicates(inplace=True)
pivot_df = history_df.pivot(index="Date", columns="Instrument", values="MidYield")

In [10]:
pivot_df.to_csv(save_as)